In [1]:
import duckdb

con = duckdb.connect()

path = "../../source_database/seed_data/suppliers.csv"

In [2]:
con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").show()

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ supplier_id   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ supplier_code │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ supplier_name │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ country       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ contact_email │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ payment_terms │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ is_own_brand  │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ is_active     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ created_at    │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴───

In [3]:
# ── Completeness Audit: how many NULLs does each column have? ──

# Step 1: Get the list of column names automatically from the file.
# This way the same audit code works on ANY csv
columns_df = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
column_names = columns_df['column_name'].tolist()

# Step 2: Get the total row count once, so we can turn null counts into %.
total_rows = con.sql(f"SELECT COUNT(*) AS cnt FROM read_csv_auto('{path}')").df()['cnt'][0]
print(f"Total rows in file: {total_rows}")

# Step 3: For each column, ask DuckDB:
#   COUNT(*)        -> counts every row (NULL or not)
#   COUNT(col_name) -> counts only NON-null rows for that column
#   so COUNT(*) - COUNT(col_name) = how many rows are NULL in that column
null_report = []
for col in column_names:
    result = con.sql(f"""
        SELECT
            '{col}' AS column_name,
            COUNT(*) - COUNT({col}) AS null_count,
            ROUND(100.0 * (COUNT(*) - COUNT({col})) / COUNT(*), 2) AS null_percentage
        FROM read_csv_auto('{path}')
    """).df()
    null_report.append(result)

# Step 4: Stack all the per-column results into one table and show it.
import pandas as pd
null_report_df = pd.concat(null_report, ignore_index=True)
null_report_df

Total rows in file: 20


,column_name,null_count,null_percentage
0,supplier_id,0,0.0
1,supplier_code,0,0.0
2,supplier_name,0,0.0
3,country,0,0.0
4,contact_email,0,0.0
5,payment_terms,0,0.0
6,is_own_brand,0,0.0
7,is_active,0,0.0
8,created_at,0,0.0


In [4]:
# Uniqueness

for col in ['supplier_id', 'supplier_code', 'supplier_name', 'contact_email']:
    dupes = con.sql(f"""
        SELECT {col}, COUNT(*) AS n FROM read_csv_auto('{path}')
        GROUP BY {col} HAVING COUNT(*) > 1
    """).df()
    print(f"Duplicate {col}: {len(dupes)}")
    print(dupes)
    print()

Duplicate supplier_id: 0
Empty DataFrame
Columns: [supplier_id, n]
Index: []

Duplicate supplier_code: 0
Empty DataFrame
Columns: [supplier_code, n]
Index: []

Duplicate supplier_name: 0
Empty DataFrame
Columns: [supplier_name, n]
Index: []

Duplicate contact_email: 0
Empty DataFrame
Columns: [contact_email, n]
Index: []



In [5]:
# Referential integrity (reverse direction: does products.csv point to real suppliers?

# WHY: suppliers is the PARENT table — check that every supplier_id used
# elsewhere (e.g. products.csv) actually exists here. Written defensively:
# only runs if products.csv actually has a supplier_id column.

products_path = "../../source_database/seed_data/products.csv"
products_cols = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{products_path}')").df()['column_name'].tolist()

if 'supplier_id' in products_cols:
    orphan_supplier_refs = con.sql(f"""
        SELECT COUNT(*) FROM read_csv_auto('{products_path}') p
        LEFT JOIN read_csv_auto('{path}') s ON p.supplier_id = s.supplier_id
        WHERE s.supplier_id IS NULL
    """).fetchone()[0]
    print(f"products.csv rows with a supplier_id that doesn't exist in suppliers.csv: {orphan_supplier_refs}")
else:
    print("products.csv has no 'supplier_id' column — skip, or point this at whichever table references suppliers.")

products.csv rows with a supplier_id that doesn't exist in suppliers.csv: 0


In [6]:
# supplier_code must follow the SUP0XX pattern AND match supplier_id

# WHY: same structural rule as store_code in stores.csv — SUP001 should
# always equal supplier_id 1. If a business-facing report joins on
# supplier_code instead of supplier_id and they've drifted apart, it
# silently attributes purchases to the wrong supplier.

code_mismatch = con.sql(f"""
    SELECT supplier_id, supplier_code,
           TRY_CAST(regexp_extract(supplier_code, '(\\d+)', 1) AS INTEGER) AS code_number
    FROM read_csv_auto('{path}')
    WHERE supplier_code !~ '^SUP\\d{{3}}$'
       OR TRY_CAST(regexp_extract(supplier_code, '(\\d+)', 1) AS INTEGER) != supplier_id
""").df()
print(f"supplier_code not matching 'SUP0XX' pattern or not matching supplier_id: {len(code_mismatch)}")
print(code_mismatch)

supplier_code not matching 'SUP0XX' pattern or not matching supplier_id: 0
Empty DataFrame
Columns: [supplier_id, supplier_code, code_number]
Index: []


In [7]:
# categorical columns: distinct values, check for typos

# WHY: shows every distinct value in each categorical column so you can
# eyeball it for case mismatches or typos (e.g. "UK" vs "U.K." vs "United
# Kingdom" would silently split into 3 groups in any GROUP BY report).

for col in ['country', 'payment_terms', 'is_own_brand', 'is_active']:
    print(f"\n=== distinct values: {col} ===")
    print(con.sql(f"""
        SELECT {col}, COUNT(*) AS n
        FROM read_csv_auto('{path}')
        GROUP BY {col}
        ORDER BY n DESC
    """).df())


=== distinct values: country ===
       country   n
0           UK  11
1  Netherlands   6
2       France   2
3        Spain   1

=== distinct values: payment_terms ===
   payment_terms   n
0             30  11
1             60   3
2             14   3
3             45   3

=== distinct values: is_own_brand ===
   is_own_brand   n
0         False  19
1          True   1

=== distinct values: is_active ===
   is_active   n
0       True  20


In [8]:
# contact_email format validity

# WHY: a malformed email means automated supplier communication (PO
# confirmations, invoice queries) silently fails to deliver.

bad_email = con.sql(f"""
    SELECT supplier_id, supplier_code, contact_email
    FROM read_csv_auto('{path}')
    WHERE contact_email !~ '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{{2,}}$'
""").df()
print(f"Malformed email addresses: {len(bad_email)}")
print(bad_email)

Malformed email addresses: 0
Empty DataFrame
Columns: [supplier_id, supplier_code, contact_email]
Index: []


In [9]:
# country vs email domain mismatch 

# WHY: every email in this file ends in .co.uk, including suppliers whose
# country is listed as Netherlands, Spain, or France. That's not necessarily
# wrong (a foreign supplier could have a UK sales office), but it's unusual
# enough across EVERY non-UK row that it's worth a human decision: is
# `country` wrong, or is `contact_email` wrong, or is this legitimately how
# the business operates?

country_email_mismatch = con.sql(f"""
    SELECT supplier_id, supplier_code, supplier_name, country, contact_email
    FROM read_csv_auto('{path}')
    WHERE country != 'UK' AND contact_email ~ '\\.co\\.uk$'
""").df()
print(f"Non-UK country but .co.uk email domain: {len(country_email_mismatch)}")
print(country_email_mismatch)

Non-UK country but .co.uk email domain: 0
Empty DataFrame
Columns: [supplier_id, supplier_code, supplier_name, country, contact_email]
Index: []


In [10]:
# payment_terms validity

# WHY: payment_terms (days) should be a small positive number. A negative,
# zero, or absurdly large value (e.g. 10,000 days) usually signals a unit
# mix-up or fat-fingered entry, and it directly affects cash-flow/AP
# reporting if it's wrong.

bad_terms = con.sql(f"""
    SELECT supplier_id, supplier_code, payment_terms
    FROM read_csv_auto('{path}')
    WHERE payment_terms <= 0 OR payment_terms > 180
""").df()
print(f"payment_terms outside a sane 1-180 day range: {len(bad_terms)}")
print(bad_terms)

payment_terms outside a sane 1-180 day range: 0
Empty DataFrame
Columns: [supplier_id, supplier_code, payment_terms]
Index: []


In [11]:
# is_own_brand sense check

# WHY: is_own_brand should be TRUE for exactly the supplier(s) representing
# NorthPeak's own-label products — a business rule, not a generic data
# check. More than one, or zero, is worth investigating.

own_brand_rows = con.sql(f"""
    SELECT supplier_id, supplier_code, supplier_name, is_own_brand
    FROM read_csv_auto('{path}')
    WHERE is_own_brand = TRUE
""").df()
print(f"Suppliers flagged is_own_brand = TRUE: {len(own_brand_rows)}")
print(own_brand_rows)

Suppliers flagged is_own_brand = TRUE: 1
   supplier_id supplier_code        supplier_name  is_own_brand
0           16        SUP016  NorthPeak Own Brand          True


In [12]:
# distribution: suppliers by country and payment terms

# WHY: a sense check, not an error check — see if the supplier base is
# concentrated or spread out in a way that matches what you'd expect for
# a UK retailer's supply chain.

print("=== Suppliers per country ===")
print(con.sql(f"SELECT country, COUNT(*) AS n FROM read_csv_auto('{path}') GROUP BY country ORDER BY n DESC").df())

print("\n=== Suppliers per payment_terms ===")
print(con.sql(f"SELECT payment_terms, COUNT(*) AS n FROM read_csv_auto('{path}') GROUP BY payment_terms ORDER BY payment_terms").df())

=== Suppliers per country ===
       country   n
0           UK  11
1  Netherlands   6
2       France   2
3        Spain   1

=== Suppliers per payment_terms ===
   payment_terms   n
0             14   3
1             30  11
2             45   3
3             60   3


In [13]:
# created_at sense check

# WHY: every row has the same created_at (2020-01-01) — same pattern as
# stores.csv. Not an error, but confirms this is a one-time bulk load with
# no per-supplier onboarding history, which matters if you're about to
# build change-tracking on top of this table.

distinct_created = con.sql(f"""
    SELECT created_at, COUNT(*) AS n
    FROM read_csv_auto('{path}')
    GROUP BY created_at
""").df()
print("=== distinct created_at values ===")
print(distinct_created)

=== distinct created_at values ===
  created_at   n
0 2020-01-01  20


In [14]:
# Sentinel/Default value usage:

# Part 1: scan for known/common sentinel patterns
SENTINEL_VALUES = [
    '-1', '0', '999', '9999', '99999', '-999',
    'NA', 'N/A', 'NULL', 'NONE', 'UNKNOWN', 'UNK', 'TBD', '-', '?',
    '1900-01-01', '1970-01-01', '9999-12-31', '0001-01-01'
]

def sentinel_scan(con, path, sentinel_values):
    schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    columns = schema['column_name'].tolist()
    values_sql = ", ".join(f"'{v.upper()}'" for v in sentinel_values)

    all_rows = []
    for col in columns:
        df = con.sql(f"""
            SELECT
                '{col}' AS column_name,
                "{col}" AS value,
                COUNT(*) AS row_count
            FROM read_csv_auto('{path}', all_varchar=True)
            WHERE upper(TRIM("{col}")) IN ({values_sql})
            GROUP BY "{col}"
            ORDER BY row_count DESC
        """).df()
        all_rows.append(df)

    return pd.concat(all_rows, ignore_index=True)

sentinel_report = sentinel_scan(con, path, SENTINEL_VALUES)
sentinel_report


,column_name,value,row_count


In [15]:
# Format consistency: Flag columns that contain mixed data types
def format_consistency_check(con, path):
    # Step 1: get every column name in the file
    schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    columns = schema['column_name'].tolist()

    all_rows = []
    for col in columns:
        # Step 2: for THIS column, classify every value into a bucket
        breakdown = con.sql(f"""
            SELECT
                '{col}' AS column_name,
                CASE
                    WHEN "{col}" IS NULL OR TRIM("{col}") = '' THEN 'NULL/EMPTY'
                    WHEN TRY_CAST("{col}" AS BIGINT) IS NOT NULL THEN 'INTEGER'
                    WHEN TRY_CAST("{col}" AS DOUBLE) IS NOT NULL THEN 'DECIMAL'
                    WHEN TRY_CAST("{col}" AS DATE) IS NOT NULL THEN 'DATE'
                    WHEN TRY_CAST("{col}" AS TIMESTAMP) IS NOT NULL THEN 'TIMESTAMP'
                    WHEN lower("{col}") IN ('true','false') THEN 'BOOLEAN'
                    ELSE 'TEXT'
                END AS detected_type,
                COUNT(*) AS row_count
            FROM read_csv_auto('{path}', all_varchar=True)
            GROUP BY detected_type
            ORDER BY row_count DESC
        """).df()
        all_rows.append(breakdown)

    result = pd.concat(all_rows, ignore_index=True)

    # Step 3: flag a column only if it has 2+ real (non-null) types
    n_types = result[result['detected_type'] != 'NULL/EMPTY'].groupby('column_name')['detected_type'].transform('nunique')
    result['is_mixed'] = (n_types.reindex(result.index).fillna(0) > 1)
    return result

report = format_consistency_check(con, path)
report                       # full breakdown — bare, no print(), so it renders as a table

,column_name,detected_type,row_count,is_mixed
0,supplier_id,INTEGER,20,False
1,supplier_code,TEXT,20,False
2,supplier_name,TEXT,20,False
3,country,TEXT,20,False
4,contact_email,TEXT,20,False
5,payment_terms,INTEGER,20,False
6,is_own_brand,BOOLEAN,20,False
7,is_active,BOOLEAN,20,False
8,created_at,DATE,20,False
